# with_steps + Metrics/Tracing Demo

Shows how to attach metrics/error counts while running `StepSpec` sequences via `AgentBuilder.with_steps`.

In [ ]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter, StepSpec
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.observability import Tracer

def stub_llm(prompt: str) -> str:
    return f"[stub llm] {prompt[:80]}"

try:
    llm_adapter = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm_adapter = FunctionAdapter(stub_llm)

tracer = Tracer(run_id="with-steps-demo")

def make_step(tag: str):
    def step(ctx: Context) -> AgentStep:
        with tracer.span(f"step:{tag}"):
            tracer.metric("step.start", 1, tag=tag)
            content = f"{tag}:{ctx.scratch.get('batch_item', '')}:{ctx.goal}"
            return AgentStep(out_messages=[Message(role=tag, content=content)], state_updates={tag: True})
    return step

steps = [
    StepSpec(name="prep", fn=make_step("prep")),
    StepSpec(name="batch", fn=make_step("batch"), batch_key="items", parallel=True, isolate_state=True),
    StepSpec(name="finish", fn=make_step("finish")),
]

agent = AgentBuilder(name="with-steps", role="orchestrator").with_llm(llm_adapter).with_steps(steps).build()
ctx = Context(goal="metric demo", scratch={"items": [1,2,3]})
result = agent.run(ctx)

events = tracer.events
([m.content for m in result.out_messages], [(e.kind, e.payload.get('name', ''), e.payload.get('tag')) for e in events])